# Registration with ICP

In this exercise, we will do rigid registration of two meshes. Between the two input meshes, there exists a rigid transformation. This transformation can be parameterized using a rotation matrix with three degrees of freedom and a translation vector with three degrees of freedom.
In the notebook below, meshes are loaded in, and correspondence points are given on both. 
Your tasks will be to implement Kabsch' algorithm to solve the orthogonal Procrustes problem, and to implement the basic ICP algorithm. In order to make this computationally feasibly, you should use a kDtree, as implemented in SciPy.


In [ ]:
from pygel3d import hmesh, jupyter_display as jd
import numpy as np
from scipy.spatial import KDTree
from numpy.linalg import norm,det
from scipy.linalg import eigh,solve,lstsq,svd
from itertools import permutations,combinations
from math import tan,acos,cos,sqrt,exp
from random import random
from time import sleep
import plotly.offline as py
import plotly.graph_objs as go
jd.set_export_mode(True)

We make an alternative function sligtly, to be able to visualize two manifolds in the same viewer.

In [ ]:
def gelmesh_to_plotyly(m, c):
    xyz = np.array([ p for p in m.positions()])
    m_tri = hmesh.Manifold(m)
    hmesh.triangulate(m_tri)
    ijk = np.array([[ idx for idx in m_tri.circulate_face(f,'v')] for f in m_tri.faces()])
    mesh = go.Mesh3d(x=xyz[:,0],y=xyz[:,1],z=xyz[:,2],
            i=ijk[:,0],j=ijk[:,1],k=ijk[:,2],color=c,flatshading=True,opacity=0.50)
    return mesh

def display_meshes(m0, m1):
    mesh0 = gelmesh_to_plotyly(m0, '#dddddd')
    mesh1 = gelmesh_to_plotyly(m1, '#dd0000')
    
    mesh_data = [mesh0, mesh1]

    fig = go.FigureWidget(data=mesh_data)
    fig.layout.scene.aspectmode="data"
    py.iplot(fig)
    
def display_points(P0, P1):
    trace0 = go.Scatter3d(
        x = P0[:,0],
        y = P0[:,1],
        z = P0[:,2],
        mode = 'markers'
    )
    trace1 = go.Scatter3d(
        x = P1[:,0],
        y = P1[:,1],
        z = P1[:,2],
        mode = 'markers'
    )
    data = [trace0, trace1]
    fig = go.Figure(data=data)
    py.iplot(fig, filename='Annotation Points')

Load data and visualize. ``wolf0det.obj`` is a lower resolution and transformed mesh of a wolf. ``wolf0.obj`` contains the original, higher resolution mesh.

In [ ]:
m0 = hmesh.load("wolf0det.obj")
m1 = hmesh.load("wolf0.obj")

display_meshes(m0, m1)

### Part1: Implement the Kabsch algorithm
Given two sets of corresponding points, return the optimal transformation.
Below you should implement the algorithm. The input is as follows:

- ``P0`` is the array of data points from the transformed data set. 
- ``P1`` is the array of reference points from the original data set. 

The return values are:

- ``P0m`` is the centroid of data points. 
- ``P1m`` is the centriod of reference points. 
- ``R`` is the rotation matrix. 

### Part 2: Coarse registration using manual correspondences

We define some manual correspondence points on m0 and m1. Aligning these using your compute_transform function should give a rough alignment of the ``m0`` and ``m1``. In order to apply the transformation, we need a function that transforms the vertices of a mesh. Implement this below.

In [ ]:
def compute_transform(P0, P1):
    P0m = np.mean(P0, axis=0)
    P1m = np.mean(P1, axis=0)

    P0_cent = P0 - P0m
    P1_cent = P1 - P1m

    H = P0_cent.T @ P1_cent
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    return P0m, P1m, R


def transform_mesh(m, P0m, P1m, R):
    pos = m.positions()
    pos[:] = (R @ (pos - P0m).T).T + P1m



P0 = np.array([[  20.0,  159.0,   29.0],
 [   2.0,    159.0,    36.0],
 [  11.0,  149.0,   20.0 ],
 [   2.0,  150.0,   23.0],
 [   1.0,  154.0,    7.0],
 [   1.0,   144.0,    7.0],
 [   7.0,   78.0,  -16.0],
 [ -19.0,   79.0,   -6.0 ],
 [  27.0,   17.0,   42.0],
 [   2.0,   19.0,   51.0],
 [  41.0,   31.0,  120.0]])

P1 = np.array([[  -10.0,  -50.0,   90.0 ],
 [  10.0,  -50.0,   90.0],
 [  -5.0,  -50.0,    80.0],
 [   5.0,  -50.0,   80.0],
 [   0.0,  -70.0,   70.0],
 [   0.0,  -60.0,   70.0],
 [ -10.0,  -30.0,    0.0],
 [  10.0,  -30.0,    0.0],
 [ -10.0,   50.0,    5.0],
 [  10.0,   50.0,    5.0],
 [   0.0,  100.0,   70.0]])

# Show points before transformation
#display_points(P0, P1)
# Determine transformation
p0m, p1m, R = compute_transform(P0, P1)
# Apply transformation
transform_mesh(m0, p0m, p1m, R)
# Show transformed mesh 
display_meshes(m0, m1)

## Part 3: The ICP Pipeline
Now that you have the Kabsch algorithm working, you can implement the iterative loop of ICP below.

In [ ]:
pos0 = m0.positions()
pos1 = m1.positions()
tree = KDTree(pos1)

max_iter = 50
E = np.zeros(max_iter)
err = 0.000005

for i in range(0, max_iter):
    distances, indices = tree.query(pos0, k=1)
    Q_match = pos1[indices]

    P0m, P1m, R = compute_transform(pos0, Q_match)
    pos0[:] = (R @ (pos0 - P0m).T).T + P1m    
    score = np.mean(distances**2)
    E[i] = score

    if i != 0 and E[i-1] - E[i] < err:
        break

    print(i, score) 
    

Show the registered mesh:

In [ ]:
display_meshes(m0, m1)

Plot the convergence curve:

In [ ]:
trace = go.Scatter(
    x = np.arange(1, max_iter),
    y = E
)
data = [trace]
py.iplot(data)

### Part 4: Questions

- What are the weaknesses of the ICP Algorithm?


One if the weakness of the ICP algorithme is that it can be sensitive to diffrent starting positions and can find local minimums that are not optimal, 

- Why are boundaries a problem for ICP, and what can be done?


We assume that the nearest point is important, but at the boundaries this usally breaks down and matches are not very usefull

- Does our use of the kD tree improve the precision or the performance? Why?


The kD tree improves the performance. this is becuase the tree datastructure is much more efficient for searching than doing a linear search

